# Send an entity with relations as JSON and read it back

An order points at its customer and holds its lines. In JSON a One relation is its id, with the entity when it is loaded; a Many relation is its ids, with the entities when they are loaded. The same forms are read back from JSON and from a dict, and each id takes the type of the target's id field.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [2]:
from typing import Annotated
from uuid import UUID

from pydantic import Field, ValidationError

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, BaseDetailEntity, CompositeMany, NoInverse
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [3]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Customer", domain=ShopDomain)
class CustomerCoreEntity(BaseCoreEntity):
    """A customer identified by a UUID."""

    id: UUID = Field(description="Customer identifier")
    name: str = Field(description="Customer name")


@entity(description="Order line", domain=ShopDomain)
class LineDetailEntity(BaseDetailEntity):
    """One line of an order."""

    id: str = Field(description="Line identifier")
    qty: int = Field(description="Quantity")


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    """An order with its customer and its lines."""

    id: str = Field(description="Order identifier")
    customer: Annotated[AssociationOne[CustomerCoreEntity], NoInverse()] = Field(description="Customer")
    lines: Annotated[CompositeMany[LineDetailEntity], NoInverse()] = Field(description="Lines")


@meta(description="Stores the shop entities")
@entities(
    CustomerCoreEntity,
    LineDetailEntity,
    OrderCoreEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""


CUSTOMER_ID = "00000000-0000-0000-0000-000000000001"

## Ids only

In [4]:
print("1) Ids only")
order = OrderCoreEntity(
    id="ord-1",
    customer=AssociationOne[CustomerCoreEntity](id=CUSTOMER_ID),
    lines=CompositeMany[LineDetailEntity](ids=("l1", "l2")),
)
print("   JSON:", order.model_dump(mode="json"))
print("   customer id type:", type(order.customer.id).__name__)

1) Ids only
   JSON: {'id': 'ord-1', 'customer': {'id': '00000000-0000-0000-0000-000000000001'}, 'lines': {'ids': ['l1', 'l2']}}
   customer id type: UUID


## Loaded

In [5]:
print("2) Loaded")
loaded = OrderCoreEntity(
    id="ord-2",
    customer=AssociationOne[CustomerCoreEntity](
        id=CUSTOMER_ID, entity=CustomerCoreEntity.partial(id=UUID(CUSTOMER_ID), name="Alice")
    ),
    lines=CompositeMany[LineDetailEntity](ids=("l1",), entities=(LineDetailEntity(id="l1", qty=2),)),
)
print("   JSON:", loaded.model_dump(mode="json"))

2) Loaded
   JSON: {'id': 'ord-2', 'customer': {'id': '00000000-0000-0000-0000-000000000001', 'entity': {'id': '00000000-0000-0000-0000-000000000001', 'name': 'Alice'}}, 'lines': {'ids': ['l1'], 'entities': [{'id': 'l1', 'qty': 2}]}}


## Loaded with zero lines, and not loaded

In [6]:
print("3) Loaded with zero lines, and not loaded")
none_loaded = CompositeMany[LineDetailEntity](ids=(), entities=(), entities_loaded=True)
print("   zero lines:", order.model_copy(update={"lines": none_loaded}).model_dump(mode="json")["lines"])
print("   not loaded:", order.model_copy(update={"lines": CompositeMany[LineDetailEntity](ids=())}).model_dump(mode="json")["lines"])

3) Loaded with zero lines, and not loaded
   zero lines: {'ids': [], 'entities': []}
   not loaded: {'ids': []}


## Read back

In [7]:
print("4) Read back")
back = OrderCoreEntity.model_validate_json(loaded.model_dump_json())
print("   equal:", back == loaded, "| customer:", back.customer.name, "| lines loaded:", back.lines.is_loaded)
from_dict = OrderCoreEntity.model_validate({"id": "ord-3", "customer": {"id": CUSTOMER_ID}, "lines": {"ids": ["l9"]}})
print("   from a dict:", from_dict.customer, from_dict.lines)

4) Read back
   equal: True | customer: Alice | lines loaded: True
   from a dict: AssociationOne(id=UUID('00000000-0000-0000-0000-000000000001'), id_only) CompositeMany(count=1, entities_loaded=no, n_entities=0)


## An id of the wrong type

In [8]:
print("5) An id of the wrong type")
try:
    OrderCoreEntity.model_validate({"id": "ord-4", "customer": {"id": "abc"}, "lines": {"ids": []}})
except ValidationError as exc:
    error = exc.errors()[0]
    print("   ValidationError at", error["loc"], "-", error["type"])
else:
    raise AssertionError("A customer id must be a UUID")

5) An id of the wrong type
   ValidationError at ('customer', 'id') - uuid_parsing


The key `"entity"` or `"entities"` is what says loaded, so zero loaded lines and lines never loaded stay different. Inside a target a missing field means not loaded: the customer here is a partial entity with only an id and a name.